# Adult Income: Classical ML, PyTorch and scoring

Use **Runtime → Run all** on Google Colab with CPU. Setup runs before data-library imports. No Drive mount, Kaggle token or GPU is required. Internet is needed for public source, dependencies and UCI data.

The default walkthrough reads the recorded experiment and runs tests plus the public-validation scorer. Set `REUSE_RECORDED_RUN = False` to train again. Previously observed test results are not an untouched holdout.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

print('Notebook Python:', sys.version, flush=True)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/adult_benchmark.py').is_file()), None)
if ROOT is None:
    ROOT = Path.cwd() / 'adult-income-project'
    if not (ROOT / 'src/adult_benchmark.py').is_file():
        if ROOT.exists():
            raise RuntimeError('adult-income-project exists without the source; choose an empty working directory.')
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VietAnh1803/HCMUT_ML_AdultCensusIncome_Pipeline.git', str(ROOT)], check=True)
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') is not None else False
INSTALL_DEPENDENCIES = IN_COLAB  # Set True locally only if dependencies are missing.
NOTEBOOK_REQUIREMENTS = "numpy==1.26.4; python_version < '3.13'\nnumpy==2.2.6; python_version >= '3.13'\npandas==2.3.2\nscikit-learn==1.7.2\nimbalanced-learn==0.14.1\nlightgbm==4.6.0\ntorch==2.9.1\nmatplotlib==3.10.6\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.30.1\n"
if INSTALL_DEPENDENCIES:
    if sys.version_info[:2] not in [(3, 12), (3, 13)]:
        raise RuntimeError('Use a Python 3.12 or 3.13 runtime; other versions have not been validated.')
    requirements_path = ROOT / '.cache/notebook-requirements.txt'
    requirements_path.parent.mkdir(parents=True, exist_ok=True)
    requirements_path.write_text(NOTEBOOK_REQUIREMENTS, encoding='utf-8')
    print('Installing CPU PyTorch...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--only-binary=:all:', 'torch==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True, timeout=600)
    print('Installing pinned notebook dependencies...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--only-binary=:all:', '-r', str(requirements_path)], check=True, timeout=600)
from IPython.display import display, Image
import pandas as pd
def run_script(relative, *arguments):
    subprocess.run([sys.executable, str(ROOT / relative), *map(str, arguments)], cwd=ROOT, check=True)
print('Source:', ROOT)
print('Colab:', IN_COLAB)
import importlib.metadata
print('Installed packages:', {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'torch', 'lightgbm']})

## Recorded results or fresh training
The default displays measured, shipped scores without retraining. Fresh training uses the same configurations in a new Python process, so package versions do not depend on already-imported kernel modules.

In [ ]:
REUSE_RECORDED_RUN = True
OUT = ROOT / 'reports/improved'
if not REUSE_RECORDED_RUN or not (OUT / 'metrics.json').is_file():
    OUT = ROOT / '.cache/notebook-retrain'
    run_script('src/adult_benchmark.py', '--output', OUT, '--cache', ROOT / '.cache/adult')
else:
    print('Recorded CPU experiment: this cell does not retrain.')
report = json.loads((OUT / 'metrics.json').read_text())
display(pd.Series(report['split']).to_frame('Split evidence'))

## Run all 50 automated cases
These verify implementation behavior, including scoring, leakage checks, early stopping and actual checkpoint weights.

In [ ]:
TEST_RECORD = ROOT / '.cache/notebook-test-results.json'
run_script('tools/run_tests.py', '--output', TEST_RECORD)
tests = json.loads(TEST_RECORD.read_text())
assert tests['successful'] and tests['tests_run'] == tests['passed'] == 50
print(f"AUTOMATED_TESTS_COMPLETE: {tests['passed']}/{tests['tests_run']} passed")

## Public-validation scoring demo
The logistic reference fits on training only and reads no evaluation labels. The scorer joins by ID. The 6,508-row validation labels are public and already used for selection; this is not a hidden challenge or a new holdout.

In [ ]:
DEMO = ROOT / '.cache/notebook-scoring'
run_script('src/local_evaluation.py', 'prepare', '--output', DEMO)
run_script('src/local_evaluation.py', 'reference', '--train', DEMO / 'train.csv', '--features', DEMO / 'evaluation_features.csv', '--output', DEMO / 'predictions.csv')
run_script('src/local_evaluation.py', 'score', '--labels', DEMO / 'evaluation_labels.csv', '--predictions', DEMO / 'predictions.csv', '--output', DEMO / 'score.json')
score = json.loads((DEMO / 'score.json').read_text())
assert score['n'] == 6508
display(pd.Series(score).to_frame('Fresh public-validation score'))
print('PUBLIC_SCORING_COMPLETE')

## Validation comparison
Model and threshold selection use validation, excluding the predefined controls.

In [ ]:
validation = pd.json_normalize(report['validation_experiments'])
display(validation[['name', 'family', 'validation.average_precision', 'validation.roc_auc']].sort_values('validation.average_precision', ascending=False))
display(pd.Series(report['selection']).to_frame('Frozen choices'))

## Filtered official test and three MLP seeds
These are the experiment selected above: recorded by default, freshly trained only when reuse is disabled. Scores refer to a previously observed public test, not independent confirmation.

In [ ]:
display(pd.DataFrame(report['test']).T[['accuracy', 'average_precision', 'roc_auc', 'f1_positive']])
display(pd.DataFrame(report['selected_mlp_seed_repeats'])[['seed', 'accuracy', 'average_precision', 'roc_auc']])
assert all(report['checks'].values())
print('ADULT_REPORT_VERIFIED: recorded/fresh protocol checks passed')

## Actual experiment plot
The MLP remains behind LightGBM. Accuracy differences also reflect validation-selected thresholds. One split and three seeds do not establish generalization to current populations.

In [ ]:
display(Image(filename=str(OUT / 'comparison.png')))
display(pd.Series(report['environment']).to_frame('Experiment environment'))